# NLP Tweet Sentiment Classification - Reference Solution

This notebook demonstrates the reference solution using TF-IDF + Logistic Regression for multi-class sentiment classification of tweets.

In [ ]:
import csv
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score

In [ ]:
def load_csv(path):
    with open(path, 'r', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' MENTION ', text)
    text = re.sub(r'#(\w+)', r' \1 ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_rows = load_csv('dataset/public/train.csv')
test_rows = load_csv('dataset/public/test.csv')

X_train = [preprocess_text(r['text']) for r in train_rows]
y_train = [r['sentiment'] for r in train_rows]
X_test = [preprocess_text(r['text']) for r in test_rows]
test_ids = [r['tweet_id'] for r in test_rows]

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000, sublinear_tf=True, min_df=2)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', multi_class='multinomial', solver='lbfgs'))
])

pipeline.fit(X_train, y_train)
print('Model trained.')

In [ ]:
y_pred = pipeline.predict(X_test)

with open('predictions.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['tweet_id', 'predicted_sentiment'])
    writer.writeheader()
    for tid, pred in zip(test_ids, y_pred):
        writer.writerow({'tweet_id': tid, 'predicted_sentiment': pred})

print(f'Saved {len(y_pred)} predictions to predictions.csv')